# Lección 13.6 — Hiperparámetros de generación: las perillas de la creatividad

> **Módulo 13 · Modelos generativos y LLMs** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 11.2 (tokens), Lección 12.4 (GPT: un decodificador), Lección 13.5 (cómo se entrena un LLM)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué produce **realmente** un LLM en cada paso: una lista de probabilidades para el siguiente token.
- Predecir el efecto de la **temperatura**, **top-k** y **top-p** sobre el texto generado.
- Controlar la **longitud** y usar **penalizaciones de repetición** para evitar bucles.
- Elegir una configuración razonable según la tarea (respuestas exactas vs. textos creativos).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Esta celda carga **GPT-2** (un LLM pequeño de 2019, 124 millones de parámetros) y dos funciones de ayuda. La primera vez puede tardar un poco porque descarga el modelo (~500 MB).

> 🌍 GPT-2 se entrenó casi solo con textos en **inglés**, así que le hablaremos en inglés. Las ideas son idénticas para cualquier LLM moderno (ChatGPT, Claude, Llama, Gemini…): todos tienen estas mismas perillas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
modelo = AutoModelForCausalLM.from_pretrained("gpt2").eval()

def siguiente_token(texto, temperatura=1.0, n=10):
    """Muestra los n tokens más probables para continuar el texto."""
    ids = tok(texto, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = modelo(ids).logits[0, -1]          # una puntuación por cada token del vocabulario
    probs = torch.softmax(logits / temperatura, dim=-1)
    top = torch.topk(probs, n)
    for p, i in zip(top.values, top.indices):
        print(f"{tok.decode(i)!r:>14}  {p:6.1%}  " + "█" * int(p * 100))
    return probs

def generar(texto, semilla=0, max_new_tokens=30, **perillas):
    """Continúa el texto con GPT-2 usando las perillas que le pases."""
    torch.manual_seed(semilla)                       # semilla fija: mismos dados → mismo texto
    ids = tok(texto, return_tensors="pt")
    salida = modelo.generate(**ids, max_new_tokens=max_new_tokens,
                             pad_token_id=tok.eos_token_id, **perillas)
    return tok.decode(salida[0, ids.input_ids.shape[1]:])

plt.rcParams["figure.figsize"] = (8, 3.5)
print(f"✅ Todo listo: GPT-2 con {sum(p.numel() for p in modelo.parameters()):,} parámetros "
      f"y un vocabulario de {len(tok):,} tokens")

---
## 1. 📖 La intuición: el autocompletar con dados

Cuando escribes en el celular *"Hoy hace mucho…"*, el teclado te sugiere **calor**, **frío**, **viento**. Un LLM hace exactamente eso, pero con todo su vocabulario: en cada paso calcula **una probabilidad para cada token posible** y luego **elige uno**. Después añade ese token al texto y repite.

```
"Hoy hace mucho"  ──►  🧠 LLM  ──►  calor 55 % · frío 25 % · viento 12 % · sol 5 % · sueño 3 %
                                        │
                              🎲 elegir uno  ◄── ¡aquí viven las perillas de esta lección!
                                        │
"Hoy hace mucho calor"  ──►  🧠 LLM  ──►  ...
```

La red (Módulo 12) solo produce la lista de probabilidades. **Cómo se elige** el token se llama **decodificación** (*decoding*), y se controla con unos hiperparámetros que **no cambian el modelo**: puedes girarlos en cada pregunta sin reentrenar nada.

### ✋ Ejercicio 1 — Calentamiento
Con la lista de arriba (calor 55 %, frío 25 %…):
1. Si **siempre** eliges la palabra más probable, ¿qué escribirás cada vez que alguien empiece con "Hoy hace mucho"?
2. Si en cambio **tiras un dado cargado** con esas probabilidades, ¿cuántas veces de cada 100 esperas ver "sueño"?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Siempre "calor". Es predecible y seguro, pero **monótono**: diez textos que empiecen igual serán idénticos.
2. Unas **3 de cada 100**. Rara vez, pero a veces aparece una opción inesperada. Esa es la tensión de toda la lección: **predecible vs. variado**.
</details>

---
## 2. 📖 El concepto: mira la lista real de GPT-2

Veamos qué opina GPT-2 sobre cómo continuar *"Once upon a time, there was a"* (*"Había una vez un/una…"*).

### 🔮 Ejercicio 2 — ¿Qué tan seguro está?
Antes de ejecutar:
1. ¿Cuál crees que será el token más probable?
2. ¿Qué probabilidad le darías: más del 50 %, alrededor del 20 % o menos del 10 %?

✍️ *Mis predicciones:*

In [ ]:
probs = siguiente_token("Once upon a time, there was a")

<details><summary><b>👀 Ver respuesta</b></summary>

El favorito es **" man"**, pero con apenas **≈4 %**. Le siguen " great" (≈3 %), " time", " certain", " woman"… Las 10 primeras opciones juntas no llegan ni al 20 %.

Moraleja: el modelo casi nunca está "seguro". Reparte la probabilidad entre **miles** de tokens. (Fíjate en el espacio antes de cada palabra: en GPT-2 el espacio forma parte del token, Lección 11.2.)
</details>

### ✋ Ejercicio 3 — La cola larga
¿Cuántos tokens hacen falta, empezando por el más probable, para reunir el **90 %** de la probabilidad? Ejecuta y compara las dos frases. ¿En cuál está más "seguro" el modelo?

In [ ]:
def tokens_para_cubrir(texto, porcentaje=0.9):
    ids = tok(texto, return_tensors="pt").input_ids
    with torch.no_grad():
        probs = torch.softmax(modelo(ids).logits[0, -1], dim=-1)
    ordenadas = torch.sort(probs, descending=True).values
    return int((ordenadas.cumsum(0) < porcentaje).sum()) + 1

for frase in ["Once upon a time, there was a", "The capital of France is"]:   # ✏️ prueba otras
    print(f"{frase!r:<35} → {tokens_para_cubrir(frase):>5} tokens para cubrir el 90 %")

<details><summary><b>👀 Ver respuesta</b></summary>

Para el cuento hacen falta **≈3,650 tokens**; para "The capital of France is", **≈1,500**. En ninguno de los dos casos el modelo está concentrado (GPT-2 es pequeño y ni siquiera pone " Paris" en primer lugar), pero con la capital está **menos disperso**: hay menos continuaciones razonables.

Retén esta idea: **la "anchura" de la lista cambia de una frase a otra.** Será clave para entender top-p.
</details>

---
## 3. 📖 Cómo funciona: la estrategia más simple (codiciosa) y su problema

La forma más simple de decodificar es la **codiciosa** (*greedy*): en cada paso, elegir **siempre** el token más probable. No hay dados, así que el resultado es **determinista**: siempre el mismo texto.

### 🔮 Ejercicio 4 — ¿Qué escribirá el codicioso?
Le pediremos 50 tokens a GPT-2 en modo codicioso (`do_sample=False`) a partir de *"The best thing about living in a small town is"*. ¿Qué crees que pasará con un texto tan largo?
- a) Un párrafo variado y coherente
- b) Un texto que se queda atrapado repitiendo lo mismo
- c) Palabras al azar sin sentido

✍️ *Mi apuesta:*

In [ ]:
PROMPT = "The best thing about living in a small town is"
print(generar(PROMPT, max_new_tokens=50, do_sample=False))

<details><summary><b>👀 Ver respuesta</b></summary>

**b)** *"that you can live in it for a long time. You can live in it for a long time. You can live in it for a long time…"*. El modelo cae en un **bucle**: una vez que escribió la frase, repetirla se vuelve lo más probable, y el codicioso no tiene forma de escapar.

Este es el defecto clásico de la decodificación codiciosa: **textos repetitivos y aburridos**. La solución es introducir **azar controlado**: el **muestreo** (*sampling*, `do_sample=True`).
</details>

### 📖 Muestrear = tirar el dado cargado
Con `do_sample=True`, en cada paso se elige un token **al azar, según sus probabilidades**. Ahora el texto depende de la **semilla** aleatoria: la misma semilla da el mismo texto; otra semilla, otro texto.

> ⚠️ Detalle técnico: si activas `do_sample=True` sin más, la librería `transformers` aplica por su cuenta `top_k=50` (lo verás en la Sección 5). Para ver el muestreo "puro" ponemos `top_k=0`.

In [ ]:
for semilla in [0, 1, 2]:      # ✏️ prueba otras semillas
    print(f"semilla {semilla}:", repr(generar(PROMPT, semilla=semilla, do_sample=True, top_k=0)), "\n")

### ✋ Ejercicio 5
¿Se repitió alguna vez la frase en bucle? ¿Te parecen textos "humanos"? ¿Qué pasa si vuelves a ejecutar la celda sin cambiar las semillas?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Ya no hay bucles, y los textos son variados. Pero son **algo erráticos**: empiezan bien y luego se van por las ramas (lluvias, coches, aviones…), porque de vez en cuando el dado elige un token poco probable de la cola larga y el texto se descarrila.

Si vuelves a ejecutar con las mismas semillas, obtienes **exactamente** los mismos textos: la semilla fija los dados. Necesitamos algo intermedio entre "siempre el favorito" y "cualquier cosa de la cola": eso son la **temperatura**, **top-k** y **top-p**.
</details>

---
## 4. 📖 Perilla 1: la temperatura 🌡️

La temperatura **estira o aplana** la lista de probabilidades **antes** de tirar el dado:
- **Temperatura baja** (< 1) → el favorito se vuelve todavía más favorito. Texto conservador y predecible.
- **Temperatura = 1** → las probabilidades originales del modelo.
- **Temperatura alta** (> 1) → todas las opciones se parecen más. Texto sorprendente… o disparatado.
- **Temperatura → 0** → equivale a la estrategia codiciosa.

Por dentro, el truco es una división: el modelo da una puntuación (*logit*) a cada token, se divide entre la temperatura y se aplica *softmax* (Lección 9.3) para convertirla en probabilidades.

Primero **a mano**, con el ejemplo del celular:

In [ ]:
palabras = ["calor", "frío", "viento", "sol", "sueño"]
probs_base = np.array([0.55, 0.25, 0.12, 0.05, 0.03])
logits = np.log(probs_base)                      # las "puntuaciones" que daría la red

def con_temperatura(logits, T):
    z = np.exp(logits / T)                       # dividir entre T y aplicar softmax
    return z / z.sum()

TEMPERATURAS = [0.5, 1.0, 2.0]                   # ✏️ prueba 0.1 o 5
fig, axes = plt.subplots(1, len(TEMPERATURAS), figsize=(12, 3), sharey=True)
for ax, T in zip(axes, TEMPERATURAS):
    p = con_temperatura(logits, T)
    ax.bar(palabras, p, color="tab:orange")
    ax.set_title(f"T = {T}")
    for i, v in enumerate(p): ax.text(i, v + .01, f"{v:.0%}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

### 🔮 Ejercicio 6 — Lee las barras
1. Con `T = 0.5`, ¿qué probabilidad tiene "calor"? ¿Y "sueño"?
2. Con `T = 2`, ¿cuántas veces más probable es "calor" que "sueño"? (Con `T = 1` era 0.55/0.03 ≈ 18 veces.)
3. ✏️ Pon `T = 0.1`. ¿A qué estrategia de la Sección 3 se parece?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con `T = 0.5`: calor sube a **≈79 %** y sueño baja a **≈0.2 %**. Los ricos se hacen más ricos.
2. Con `T = 2`: calor ≈37 % y sueño ≈9 %: solo **≈4 veces** más probable. La lista se aplana.
3. Con `T = 0.1`, calor tiene prácticamente el 100 %: es la estrategia **codiciosa**. Por eso muchas APIs aceptan `temperature=0` como sinónimo de "codicioso".
</details>

▶️ Ahora con GPT-2 de verdad. Mismo comienzo, tres temperaturas:

In [ ]:
for T in [0.5, 1.0, 2.0]:
    print(f"--- temperatura {T}")
    _ = siguiente_token("Once upon a time, there was a", temperatura=T, n=5)

### 🔮 Ejercicio 7 — Temperatura y texto
Vamos a generar con temperaturas **0.3, 0.7, 1.0 y 1.5** (misma semilla). Antes de ejecutar, ordénalas en tu cabeza: ¿cuál producirá un texto **sin sentido**? ¿Cuál el más **"seguro"**?

✍️ *Mis predicciones:*

In [ ]:
for T in [0.3, 0.7, 1.0, 1.5]:      # ✏️ prueba 0.1 y 3.0
    print(f"T={T}:", repr(generar(PROMPT, semilla=0, do_sample=True, top_k=0, temperature=T)), "\n")

<details><summary><b>👀 Ver respuesta</b></summary>

- **T = 0.3**: frases correctas y algo genéricas (*"that you can enjoy the city and the people you meet…"*).
- **T = 0.7**: variado y todavía coherente.
- **T = 1.0**: más original, empieza a divagar.
- **T = 1.5**: se descarrila: palabras raras y frases sin sentido (*"along gritty seven highwayave stories…"*).

Regla práctica: **subir la temperatura sube la creatividad y también el riesgo de disparates.** Con `T = 3` el texto es una sopa de letras.
</details>

---
## 5. 📖 Perilla 2: top-k ✂️

La temperatura **reduce** la probabilidad de los tokens raros, pero no la hace cero: con mala suerte, el dado aún puede caer en la cola larga. **Top-k** corta por lo sano:

> Quédate solo con los **k** tokens más probables, reparte entre ellos el 100 % y tira el dado.

Con el ejemplo del celular y `k = 2`, solo sobreviven "calor" y "frío".

In [ ]:
def top_k(probs, k):
    p = probs.copy()
    p[np.argsort(p)[::-1][k:]] = 0               # borra todo lo que no esté entre los k mejores
    return p / p.sum()                           # vuelve a repartir el 100 %

K = 2   # ✏️ prueba 1, 3 y 5
for palabra, p in zip(palabras, top_k(probs_base, K)):
    print(f"{palabra:<7} {p:6.1%}")

### 🔮 Ejercicio 8
1. Con `k = 2`, ¿qué probabilidad queda para "calor"? (pista: 0.55 / (0.55 + 0.25))
2. ¿A qué estrategia equivale `top_k = 1`?

Comprueba la 2 con GPT-2: compara `top_k=1` con el resultado codicioso de la Sección 3.

In [ ]:
for k in [1, 5, 50]:          # ✏️
    print(f"top_k={k}:", repr(generar(PROMPT, semilla=0, do_sample=True, top_k=k)), "\n")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **≈69 %** para calor y ≈31 % para frío. Viento, sol y sueño quedan en 0 %.
2. `top_k = 1` deja un único candidato: es la estrategia **codiciosa**. GPT-2 vuelve a caer en el bucle *"You can live in it for a long time…"*.

Con `k = 5` el texto es coherente pero un poco extraño (¡zombis!); con `k = 50` (el valor por defecto de `transformers`) es más variado.
</details>

### 📖 El problema de un k fijo
En el Ejercicio 3 viste que la lista **cambia de anchura** según la frase. Un `k` fijo no se adapta:
- Si la frase tiene **una** respuesta obvia ("2 + 2 ="), `k = 50` deja entrar 49 opciones absurdas.
- Si la frase admite **miles** de continuaciones ("Había una vez…"), `k = 5` deja fuera muchas opciones razonables.

Esa es la motivación de la siguiente perilla.

---
## 6. 📖 Perilla 3: top-p (muestreo de núcleo) 🎯

**Top-p** (*nucleus sampling*) no fija **cuántos** candidatos, sino **cuánta probabilidad** se conserva:

> Ordena los tokens de mayor a menor y quédate con los primeros hasta **sumar p** (por ejemplo, 90 %). Ese grupo es el **núcleo**. Tira el dado solo dentro del núcleo.

Si el modelo está seguro, el núcleo es pequeño; si duda, el núcleo crece. **Se adapta solo.**

In [ ]:
def top_p(probs, p_max):
    orden = np.argsort(probs)[::-1]
    acumulada = np.cumsum(probs[orden])
    n_quedan = np.searchsorted(acumulada, p_max) + 1      # cuántos hacen falta para llegar a p_max
    p = np.zeros_like(probs)
    p[orden[:n_quedan]] = probs[orden[:n_quedan]]
    return p / p.sum()

P_MAX = 0.9   # ✏️ prueba 0.5 y 0.99
for palabra, p in zip(palabras, top_p(probs_base, P_MAX)):
    print(f"{palabra:<7} {p:6.1%}")

### 🔮 Ejercicio 9
1. Con `p = 0.9` y la lista del celular (55 %, 25 %, 12 %, 5 %, 3 %), ¿cuántas palabras entran en el núcleo?
2. ¿Y con `p = 0.5`?

✍️ *Mis predicciones:* (luego ✏️ cambia `P_MAX` y comprueba)

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Tres**: calor + frío = 80 % todavía no llega a 90 %, así que entra viento (92 %). Quedan calor ≈60 %, frío ≈27 %, viento ≈13 %.
2. **Una**: calor solo (55 %) ya supera el 50 %. Con `p` bajo, top-p se comporta casi como el codicioso.
</details>

▶️ Con GPT-2, el núcleo del 90 % cambia muchísimo de una frase a otra (lo viste en el Ejercicio 3). Veamos el texto con tres valores de `top_p`:

In [ ]:
for p in [0.3, 0.9, 1.0]:     # ✏️
    print(f"top_p={p}:", repr(generar(PROMPT, semilla=0, do_sample=True, top_k=0, top_p=p)), "\n")

### ✋ Ejercicio 10
1. ¿Cuál de los tres textos es **idéntico** a uno que ya viste en la Sección 4? ¿Por qué?
2. ¿Cuál se parece más a una temperatura baja?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **`top_p=1.0`** da exactamente el mismo texto que **`T=1.0`** del Ejercicio 7: conservar el 100 % de la probabilidad es no cortar nada, o sea, muestreo puro con la misma semilla.
2. **`top_p=0.3`**: el núcleo es tan pequeño que solo quedan los favoritos, y el texto sale correcto pero genérico (*"that you can be yourself and not be afraid…"*), igual que con temperatura baja.

> 💡 En la práctica se combina una temperatura moderada con top-p (p. ej. `temperature=0.7, top_p=0.9`): la temperatura reparte, top-p corta la cola de disparates.
</details>

---
## 7. 📖 Perilla 4: la longitud 📏

`max_new_tokens` es el **número máximo de tokens nuevos**. El modelo se detiene antes si genera el token especial de **fin de texto** (*end-of-sequence*, EOS). Ten en cuenta que:
- Se cuenta en **tokens**, no en palabras ni caracteres.
- El corte es **brusco**: el modelo no "sabe" que le quedan pocos tokens y no redondea la frase.
- En los servicios de pago, **cada token cuesta** dinero y tiempo: más largo = más caro y más lento.

### 🔮 Ejercicio 11 — ¿Cuántos tokens son?
La frase *"The cat is sleeping on the sofa."* tiene 7 palabras. Su traducción, *"El gato está durmiendo en el sofá."*, también. ¿Cuántos tokens crees que usa GPT-2 para cada una?

✍️ *Mi apuesta:*

In [ ]:
for frase in ["The cat is sleeping on the sofa.", "El gato está durmiendo en el sofá."]:   # ✏️
    tokens = tok.tokenize(frase)
    print(f"{len(frase.split())} palabras → {len(tokens):>2} tokens: {tokens}")

print("\nCon max_new_tokens=8:", repr(generar(PROMPT, max_new_tokens=8, do_sample=False)))

<details><summary><b>👀 Ver respuesta</b></summary>

Inglés: **8 tokens**. Español: **14 tokens**, casi el doble. GPT-2 aprendió su vocabulario con textos en inglés, así que parte las palabras en español en trozos ("Ġg" + "ato", "Ġdur" + "mi" + "endo"; la "Ġ" representa el espacio y "Ã¡" es la "á" codificada). Consecuencia práctica: en un modelo así, el mismo texto en español **cuesta más tokens**, es decir, más dinero y más espacio en la ventana de contexto (Lección 13.9).

Y con `max_new_tokens=8` el texto se corta a media frase: *"that you can live in it for a"*.
</details>

---
## 8. 📖 Perilla 5: penalizaciones contra la repetición 🔁

Para romper bucles como el del codicioso existen dos perillas en `transformers`:

| Perilla | Qué hace |
|---|---|
| `repetition_penalty` | Divide la puntuación de los tokens **que ya aparecieron** en el texto. 1.0 = sin castigo; 1.2–1.3 es lo usual. |
| `no_repeat_ngram_size` | Prohíbe repetir **cualquier secuencia de n tokens** que ya aparezca. Con 3, ningún trío de tokens se repite. |

> 💡 En las APIs comerciales verás `frequency_penalty` (castiga más cuanto más veces se repitió un token) y `presence_penalty` (castiga si el token ya apareció, aunque sea una vez). Son la misma idea con otra fórmula.

### 🔮 Ejercicio 12 — Rompiendo el bucle
Usaremos el modo **codicioso** (sin dados), que antes se quedó atrapado. Con `repetition_penalty` = 1.0, 1.3 y 3.0:
1. ¿Seguirá el bucle con 1.3?
2. ¿Será muy distinto el texto con 3.0 que con 1.3?

✍️ *Mis predicciones:*

In [ ]:
for rp in [1.0, 1.3, 3.0]:     # ✏️
    print(f"repetition_penalty={rp}:", repr(generar(PROMPT, max_new_tokens=40, do_sample=False,
                                                     repetition_penalty=rp)), "\n")
print("no_repeat_ngram_size=3:", repr(generar(PROMPT, max_new_tokens=40, do_sample=False,
                                               no_repeat_ngram_size=3)))

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con **1.3** el bucle desaparece: *"that you can live there for years. You don't have to worry if your house gets burned down…"*.
2. **Sorpresa: con 3.0 el texto es idéntico al de 1.3.** La penalización solo importa cuando un token repetido estaba a punto de ganar; en este texto, una vez que el castigo fue suficiente para tomar otro camino, castigar más no cambia nada. Con otros textos un castigo exagerado **sí** estropea la gramática (prohíbe de hecho palabras necesarias como "the" o "of"), por eso se usan valores modestos.

`no_repeat_ngram_size=3` también rompe el bucle, a su manera: *"You can live there for a while and then you can move on to the next city."*
</details>

---
## 9. 📖 Medir en vez de opinar: diversidad vs. temperatura

"Este texto es más variado" es una opinión. Una métrica sencilla es la **proporción de bigramas distintos** (*distinct-2*): de todos los pares de palabras seguidas en varios textos, ¿qué fracción es única? 1.0 = nada se repite; valores bajos = texto repetitivo.

### 🔮 Ejercicio 13
Generaremos 5 textos (5 semillas) para cada temperatura y mediremos *distinct-2*. ¿Cómo crees que será la curva: sube, baja o sube y luego baja? (⏱️ tarda ≈30 s.)

✍️ *Mi predicción:*

In [ ]:
def distinct_2(textos):
    pares = []
    for t in textos:
        w = t.split()
        pares += list(zip(w, w[1:]))
    return len(set(pares)) / len(pares)

temperaturas = [0.2, 0.7, 1.0, 1.5]    # ✏️
diversidad = []
for T in temperaturas:
    textos = [generar(PROMPT, semilla=s, do_sample=True, top_k=0, temperature=T) for s in range(5)]
    diversidad.append(distinct_2(textos))
    print(f"T={T}: distinct-2 = {diversidad[-1]:.2f}")

plt.plot(temperaturas, diversidad, "o-"); plt.xlabel("temperatura"); plt.ylabel("distinct-2")
plt.title("Más temperatura → más variedad (¿y más sentido?)"); plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

**Sube** sin parar: ≈0.68 con T = 0.2, ≈0.90 con 0.7, ≈0.97 con 1.0 y 1.00 con 1.5. La diversidad es fácil de conseguir: basta subir la temperatura.

Pero ojo: con T = 1.5 la diversidad es perfecta… y el texto no tiene sentido (Ejercicio 7). **Ninguna métrica sola basta**: en la práctica se mide diversidad *y* calidad (coherencia, exactitud, preferencias humanas), y se busca el punto intermedio. Es la misma idea de "equilibrio" que viste con el sobreajuste, en otra forma.
</details>

---
## 10. 🎛️ Tabla de hiperparámetros de generación

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `temperature` | Cuánto se aplana la lista de probabilidades | Más variado y creativo → riesgo de disparates | Más predecible → riesgo de repetición (0 = codicioso) | 0.7 (chat) · 0–0.3 (datos, código) · 0.9–1.1 (creativo) |
| `top_k` | Cuántos candidatos sobreviven | Más opciones (k grande ≈ sin filtro) | Más conservador (1 = codicioso) | 40–50, o desactivado (0) si usas top-p |
| `top_p` | Qué probabilidad acumulada sobrevive (núcleo) | Más opciones de la cola | Más conservador | 0.9–0.95 |
| `max_new_tokens` | Longitud máxima de la respuesta | Respuestas más largas, más caras y lentas | Cortes a media frase | Lo que necesite la tarea (+ margen) |
| `repetition_penalty` | Castigo a tokens ya usados | Menos repetición → riesgo de gramática rara | Más riesgo de bucles (1.0 = nada) | 1.1–1.3 |
| `no_repeat_ngram_size` | Prohíbe repetir secuencias de n tokens | Menos restrictivo | Muy restrictivo (con 1 no repite ni una palabra) | 0 (desactivado) o 3–4 |
| `do_sample` | ¿Tirar dados o elegir el favorito? | — | — | `True` para texto natural; `False` para reproducibilidad total |
| semilla | Qué dados se tiran | — | — | Fíjala para reproducir resultados |

### 🧠 Recetas rápidas
- **Extraer datos, responder con hechos, escribir código:** temperatura 0–0.3. Quieres la respuesta más probable, no creatividad.
- **Asistente conversacional:** temperatura ≈0.7 con top-p ≈0.9.
- **Lluvia de ideas, cuentos, eslóganes:** temperatura 0.9–1.1, y genera **varias** opciones.

> ⚠️ Mueve **una perilla a la vez** (normalmente temperatura **o** top-p). Si subes las dos, no sabrás cuál causó el cambio.

---
## 11. 🚨 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Frases que se repiten en bucle | Decodificación codiciosa o temperatura muy baja | Muestrear (`do_sample=True`), `repetition_penalty≈1.2`, `no_repeat_ngram_size=3` |
| Texto sin sentido, palabras inventadas | Temperatura alta y sin filtro de cola | Bajar temperatura, usar `top_p=0.9` |
| Respuesta cortada a media frase | `max_new_tokens` muy pequeño | Subirlo, o pedir respuestas más breves en el prompt |
| "No puedo reproducir el resultado de ayer" | Muestreo sin semilla fija | Fijar semilla, o usar temperatura 0 |
| El modelo inventa datos con temperatura 0 | **La temperatura no arregla el conocimiento**: el favorito también puede ser falso | Darle la información (RAG, Lección 13.8); ver Lección 13.9 |
| Gramática rara, evita palabras comunes | Penalización de repetición exagerada | Bajarla a 1.1–1.3 |

### 🔮 Ejercicio 14 — ¿Temperatura 0 = verdad?
Un compañero dice: *"Para que el modelo no se equivoque en fechas, pongo la temperatura en 0"*. Pregúntale a GPT-2 en modo codicioso en qué año llegó el ser humano a la Luna. ¿Acertará?

✍️ *Mi apuesta:*

In [ ]:
print(repr(generar("The first human landed on the Moon in the year", max_new_tokens=8, do_sample=False)))

<details><summary><b>👀 Ver respuesta</b></summary>

GPT-2 contesta con total "seguridad" un año **absurdo**: *" 1620."* (la respuesta correcta es 1969). Con temperatura 0 obtienes siempre la opción **favorita** del modelo, pero si el modelo no sabe el dato, su favorita también es falsa. Las perillas de generación controlan **el estilo** (variado vs. predecible), no **la verdad**. Para eso hacen falta otras herramientas: darle fuentes (RAG, Lección 13.8) y verificar (Lección 13.9).
</details>

---
## 12. 📋 Ficha: las perillas de generación

| | Decodificación de un LLM |
|---|---|
| **Qué es** | La regla para elegir el siguiente token a partir de las probabilidades que da el modelo |
| **Intuición** | Un dado cargado que puedes cargar todavía más (T baja) o igualar (T alta), y al que puedes quitarle caras (top-k, top-p) |
| **Se ajusta…** | En cada llamada, **sin reentrenar** el modelo |
| **Perillas clave** | `temperature`, `top_p`, `top_k`, `max_new_tokens`, `repetition_penalty` |
| **✅ Sirve para** | Equilibrar creatividad y precisión, evitar bucles, controlar costo y longitud |
| **❌ No sirve para** | Hacer que el modelo sepa algo que no sabe ni que siga instrucciones que no aprendió |
| **Úsalo así** | Empieza con T ≈ 0.7 y top-p ≈ 0.9; baja la temperatura para tareas exactas y súbela para tareas creativas |

## 13. 🧠 Resumen en 5 puntos
1. En cada paso, un LLM produce **probabilidades para todo su vocabulario**; la **decodificación** decide qué token se escribe.
2. **Codicioso** (siempre el favorito) es determinista pero cae en **bucles**; **muestrear** da variedad pero puede descarrilarse.
3. La **temperatura** aplana (alta) o afila (baja) la lista; **top-k** y **top-p** cortan la cola de opciones absurdas, y top-p se adapta a cada frase.
4. `max_new_tokens` limita la longitud (se cuenta en tokens, y el corte es brusco); las **penalizaciones** de repetición rompen bucles.
5. Estas perillas controlan el **estilo**, no la **verdad**: con temperatura 0 el modelo también puede inventar.

---
## 14. 🏋️ Ejercicios finales

### 14.1 Repaso rápido (opción múltiple)

**1.** Tu chatbot de soporte a veces responde con frases raras e inventa palabras. ¿Qué cambio tiene **más** sentido?
- a) Subir la temperatura de 1.2 a 1.6 · b) Bajar la temperatura de 1.2 a 0.5 · c) Subir `max_new_tokens` · d) Cambiar la semilla

**2.** ¿Qué configuración equivale a la decodificación **codiciosa**?
- a) `top_p=1.0` · b) `temperature=2.0` · c) `top_k=1` · d) `repetition_penalty=1.0`

**3.** Con las probabilidades 40 %, 30 %, 20 %, 10 % y `top_p=0.75`, ¿cuántos tokens quedan en el núcleo?
- a) 1 · b) 2 · c) 3 · d) 4

**4.** ¿Por qué top-p suele preferirse a top-k?
- a) Es más rápido · b) Adapta el número de candidatos a lo seguro que está el modelo · c) Elimina las alucinaciones · d) No necesita semilla

**5.** Generas con `do_sample=True, temperature=0.8` y la semilla fija en 42. Vuelves a ejecutar con la misma semilla. El texto será…
- a) Distinto · b) Idéntico · c) Más corto · d) Más largo

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Palabras inventadas y frases raras = demasiada aleatoriedad. Bajar la temperatura (o añadir top-p) lo arregla; a) lo empeora.
2. **c)** Con un solo candidato no hay nada que sortear. (Temperatura → 0 también.)
3. **c)** 40 % → 70 % (aún no llega a 75 %) → 90 %: entran 3.
4. **b)** Si el modelo está seguro, el núcleo es pequeño; si duda, crece. Top-k usa el mismo número siempre.
5. **b)** La semilla fija los "dados": mismo modelo, mismo prompt, mismas perillas y misma semilla → mismo texto.
</details>

### 14.2 Preguntas para pensar (sin ejecutar)

**A.** ¿Por qué el modo codicioso tiende a caer en bucles? Piensa en qué pasa con las probabilidades una vez que una frase ya apareció una vez en el texto.

**B. Caso de análisis 💼** Una aseguradora usa un LLM para dos tareas: (1) extraer el número de póliza y la fecha de un correo y (2) redactar tres borradores de una carta amable para un cliente molesto. Propón una configuración de perillas para cada tarea y justifícala. ¿Usarías la misma semilla para los tres borradores?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** El modelo aprendió que los textos suelen ser coherentes con lo que ya dijeron, así que una frase que ya está en el texto se vuelve **más** probable de repetir. El codicioso siempre elige la opción más probable, así que, una vez dentro del bucle, cada repetición refuerza la siguiente y no hay ningún dado que lo saque.

**B.** Una respuesta razonable:
- **Extracción:** `temperature=0` (o codicioso), `max_new_tokens` pequeño. Hay una única respuesta correcta y quieres que sea reproducible.
- **Cartas:** `temperature≈0.8–1.0`, `top_p≈0.9`, quizá `repetition_penalty≈1.1`. Quieres variedad entre borradores.
- **No**: con la misma semilla, las tres cartas serían idénticas. Usa tres semillas distintas (o ninguna fija) y guarda cuál usaste para cada borrador si necesitas reproducirlo.
</details>

### 14.3 🧩 Práctica integradora: "Lluvia de nombres para un café"

Una cafetería nueva te pide **ideas de nombres de bebidas**. Usarás GPT-2 con un prompt de **ejemplos** (*few-shot*, Lección 13.8): le mostramos una lista y le pedimos que la continúe. Lo harás en 5 pasos cortos. **No avances sin revisar cada respuesta.**

**Paso 1 ▶️ — El prompt.** Ejecuta para verlo.

In [ ]:
MENU = """Moonbeam Coffee - Drink menu
1. Midnight Mocha
2. Caramel Cloud Latte
3. Vanilla Starlight
4."""
print(MENU)

**Paso 2 🔮 — El codicioso.** Generaremos 40 tokens sin muestrear. ¿Qué esperas: nombres creativos, o algo repetitivo?

In [ ]:
print(generar(MENU, max_new_tokens=40, do_sample=False))

<details><summary><b>👀 Ver respuesta</b></summary>

Sigue el formato de lista (el prompt de ejemplos funciona)… pero escribe **"Vanilla Ice Cream" siete veces seguidas**. Es el bucle de la Sección 3 en acción: el codicioso siempre toma el camino más trillado. Para una lluvia de ideas no sirve.
</details>

**Paso 3 ✏️ — Tres temperaturas.** Ejecuta con 0.5, 1.0 y 1.5 (misma semilla, `top_k=0`). ¿Cuál da nombres que de verdad pondrías en un menú?

In [ ]:
for T in [0.5, 1.0, 1.5]:     # ✏️
    print(f"--- T={T}\n4." + generar(MENU, semilla=0, max_new_tokens=40, do_sample=True, top_k=0, temperature=T))

<details><summary><b>👀 Ver respuesta</b></summary>

- **T = 0.5**: respeta la lista, pero se obsesiona con la vainilla (*Vanilla Cookie, Vanilla Milk, Vanilla Bean…*): poco original.
- **T = 1.0**: aparecen nombres originales (*Jamaican Sandalwood Coffee, Honeyden Blast*), pero tras tres nombres **abandona el formato** y se pone a hablar de canoas.
- **T = 1.5**: sopa de palabras (*"Goldbacks Skooter Allegions…"*).

Ninguna es perfecta: con poca temperatura falta creatividad y con mucha se pierde el control. Falta una herramienta para cortar la cola de disparates sin perder variedad.
</details>

**Paso 4 ✏️ — Añade top-p.** Ahora deja `temperature=1.0` y recorta la cola con `top_p=0.9`. Genera con 3 semillas.

In [ ]:
for semilla in [0, 1, 2]:     # ✏️
    print(f"--- semilla {semilla}\n4." + generar(MENU, semilla=semilla, max_new_tokens=40,
                                                 do_sample=True, top_k=0, temperature=1.0, top_p=0.9))

<details><summary><b>👀 Ver respuesta</b></summary>

Con la **semilla 0** sale una lista completa y variada (*Vanilla Blast, Mango Cold Coffee & Smoke Blend…*); fíjate en que empieza igual que con `T=1.0` sin top-p, pero ya no se desvía hacia las canoas. La **semilla 2** también respeta el formato, con nombres atrevidos (*Crazy Magoo Heart, Fat Friday Boogaloo*). La **semilla 1**, en cambio, rompe el formato y amontona sabores en una sola línea.

Conclusión: top-p reduce los disparates, pero no los elimina, y cada semilla da un resultado distinto. Esa es la receta típica para tareas creativas: **temperatura moderada-alta + top-p + varias semillas + un humano que elige**.
</details>

**Paso 5 ✋ — Tu recomendación.** Escribe la configuración que le recomendarías a la cafetería (perillas + cuántas listas generar) y una frase que explique, sin tecnicismos, por qué no basta con generar una sola lista.

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Ejemplo: *"`temperature=1.0`, `top_p=0.9`, `max_new_tokens≈40`, y generar 5–10 listas con semillas distintas."* Explicación para la cafetería: *"El programa funciona como una lluvia de ideas: cada vez propone nombres distintos, algunos buenos y otros malos. Le pediremos muchas propuestas y ustedes eligen las mejores."*

Fíjate en que las perillas **no** convierten a GPT-2 en un experto en marketing: solo controlan cuánto se arriesga. La calidad de base viene del modelo y del prompt (Lecciones 13.5 y 13.8).
</details>

---
## 📚 Para profundizar
- 📄 **Artículo:** Holtzman et al. (2019), *The Curious Case of Neural Text Degeneration*: el artículo que propuso top-p y explicó los bucles del codicioso.
- 🤗 **Documentación:** Hugging Face, guía *Text generation strategies* (búsqueda por haces, muestreo, penalizaciones).
- 🎮 **Interactivo:** en cualquier "playground" de un proveedor de LLMs puedes mover la temperatura y top-p y ver el efecto al instante.

---
⬅️ **Anterior:** [13.5 Cómo se entrena un LLM: pre-entrenamiento, ajuste por instrucciones, RLHF/RLAIF](13.5-como-se-entrena-un-llm.ipynb) · ➡️ **Siguiente:** [13.7 Fine-tuning eficiente: LoRA y cuantización (intuición)](13.7-lora-y-cuantizacion.ipynb)